In [2]:
# importing relevant python libraries
import pandas as pd
import numpy as np

In [3]:
# setting up stats needed (cleaned up)
STAGE_ORDER = ['page_view', 'add_to_cart', 'checkout_start', 'payment_info', 'purchase']
STAGE_RANK = {s: i for i, s in enumerate(STAGE_ORDER)}

data = pd.read_csv("site_data.csv")
data["event_date"] = pd.to_datetime(data["event_date"])
data["stage_rank"] = data["event_type"].map(STAGE_RANK)

visitors = data.groupby("user_id").agg(
    traffic_source=("traffic_source", "first"),
    product_id=("product_id", "first"),
    first_seen=("event_date", "min"),
    furthest_stage=("stage_rank", "max"),
).reset_index()

order_value = data.loc[data.event_type == "purchase"].groupby("user_id")["amount"].sum()
visitors["order_value"] = visitors["user_id"].map(order_value).fillna(0)
visitors["converted"] = (visitors["furthest_stage"] == 4).astype(int)

channel_stats = visitors.groupby("traffic_source").agg(
    visitors=("user_id", "count"),
    conversions=("converted", "sum"),
    revenue=("order_value", "sum"),
).reset_index()
channel_stats["conversion_rate"] = channel_stats.conversions / channel_stats.visitors
channel_stats["aov"] = channel_stats.traffic_source.map(
    visitors[visitors.converted == 1].groupby("traffic_source")["order_value"].mean()
)
channel_stats["revenue_per_visitor"] = channel_stats.conversion_rate * channel_stats.aov

In [4]:
# building usable tools
rev_per_visitor = dict(zip(channel_stats.traffic_source,
                           channel_stats.revenue_per_visitor))

current_counts = dict(zip(channel_stats.traffic_source,
                          channel_stats.visitors))

def reallocate(counts, from_ch, to_ch, pct):
    """
    A channel to simulate a movement of traffic from one source to another.

    Parameters:
    ~ counts ~ the traffic source count.
    ~ from_ch ~ the channel of which traffic comes from.
    ~ to_ch ~ the channel of which traffic is going to.
    ~ pct ~ percantage of traffic being moved.

    Returns:
    A new set of traffic proportions.
    """
    new_counts = counts.copy()
    n = int(new_counts[from_ch] * pct)
    new_counts[from_ch] -= n
    new_counts[to_ch] += n
    return new_counts

def projected_revenue(counts):
    """
    A simple length of calculating revenue from the counts.

    Parameters:
    ~ counts ~ the traffic source count.

    Returns:
    The revenue.
    """
    return sum(counts[c] * rev_per_visitor[c] for c in counts)

In [5]:
# creating a scenario of which change is implemented
current_rev = projected_revenue(current_counts)

scenario = reallocate(current_counts, "social", "email", 0.3)
scenario_rev = projected_revenue(scenario)

print("Current Revenue:", current_rev)
print("Scenario Revenue:", scenario_rev)
print("Revenue Change:", scenario_rev - current_rev)

Current Revenue: 87975.11
Scenario Revenue: 99683.24810063718
Revenue Change: 11708.138100637181


In [6]:
# creating a model that combines previous tools and removes setup
def potential_revenue(from_ch, to_ch, pct):

    STAGE_ORDER = [
        'page_view',
        'add_to_cart',
        'checkout_start',
        'payment_info',
        'purchase'
    ]

    STAGE_RANK = {
        s: i for i, s in enumerate(STAGE_ORDER)
    }

    data = pd.read_csv("site_data.csv")

    data["event_date"] = pd.to_datetime(data["event_date"])

    data["stage_rank"] = data["event_type"].map(STAGE_RANK)

    visitors = data.groupby("user_id").agg(
        traffic_source=("traffic_source", "first"),
        product_id=("product_id", "first"),
        first_seen=("event_date", "min"),
        furthest_stage=("stage_rank", "max"),
    ).reset_index()


    order_value = (
        data[data.event_type == "purchase"]
        .groupby("user_id")["amount"]
        .sum()
    )

    visitors["order_value"] = (
        visitors["user_id"]
        .map(order_value)
        .fillna(0)
    )
    
    visitors["converted"] = (
        visitors["furthest_stage"] == 4
    ).astype(int)

    channel_stats = visitors.groupby("traffic_source").agg(
        visitors=("user_id", "count"),
        conversions=("converted", "sum"),
        revenue=("order_value", "sum")
    ).reset_index()

    channel_stats["conversion_rate"] = (
        channel_stats.conversions /
        channel_stats.visitors
    )

    channel_stats["aov"] = channel_stats.traffic_source.map(
        visitors[visitors.converted == 1]
        .groupby("traffic_source")["order_value"]
        .mean()
    )

    channel_stats["revenue_per_visitor"] = (
        channel_stats.conversion_rate *
        channel_stats.aov
    )

    rev_per_visitor = dict(zip(
        channel_stats.traffic_source,
        channel_stats.revenue_per_visitor
    ))

    current_counts = dict(zip(
        channel_stats.traffic_source,
        channel_stats.visitors
    ))

    current_rev = sum(
        current_counts[c] * rev_per_visitor[c]
        for c in current_counts
    )

    new_counts = current_counts.copy()

    moved = int(new_counts[from_ch] * pct)

    new_counts[from_ch] -= moved
    new_counts[to_ch] += moved

    scenario_rev = sum(
        new_counts[c] * rev_per_visitor[c]
        for c in new_counts
    )

    print("Current Revenue:", current_rev)
    print("Scenario Revenue:", scenario_rev)
    print("Revenue Change:", scenario_rev - current_rev)

In [7]:
# testing combined tool
potential_revenue("social", "email", 0.3)

Current Revenue: 87975.11
Scenario Revenue: 99683.24810063718
Revenue Change: 11708.138100637181


In [19]:
# a cleaned up version
def potential_revenue_2(from_ch, to_ch, pct):
    channel_stats = visitors.groupby("traffic_source").agg(
        visitors=("user_id", "count"),
        conversions=("converted", "sum"),
    ).reset_index()
    channel_stats["conversion_rate"] = channel_stats.conversions / channel_stats.visitors
    channel_stats["aov"] = channel_stats.traffic_source.map(
        visitors[visitors.converted == 1].groupby("traffic_source")["order_value"].mean()
    )
    channel_stats["revenue_per_visitor"] = channel_stats.conversion_rate * channel_stats.aov

    rev_per_visitor = dict(zip(channel_stats.traffic_source, channel_stats.revenue_per_visitor))
    current_counts = dict(zip(channel_stats.traffic_source, channel_stats.visitors))

    current_rev = sum(current_counts[c] * rev_per_visitor[c] for c in current_counts)

    new_counts = current_counts.copy()
    moved = int(new_counts[from_ch] * pct)
    new_counts[from_ch] -= moved
    new_counts[to_ch] += moved
    scenario_rev = sum(new_counts[c] * rev_per_visitor[c] for c in new_counts)

    print("Current Revenue:", current_rev)
    print("Scenario Revenue:", scenario_rev)
    print("Revenue Change:", scenario_rev - current_rev)

In [20]:
# testing the cleaned up version
potential_revenue_2("social", "email", 0.3)

Current Revenue: 87975.11
Scenario Revenue: 99683.24810063718
Revenue Change: 11708.138100637181
